In [2]:
import sys
import os
import time
sys.path.append(os.path.abspath("../src"))
import pandas as pd
from data_loader import *

In [3]:
all_leagues = get_all_leagues()

In [4]:
tournaments = [
    "Slam IV",
    "FISSURE PLAYGROUND 2",
    "PGL Wallachia 2025 Season 6",
    "Slam V",
    "DreamLeague Season 27",
    "BLAST Slam VI",
    "DreamLeague Season 28",
]

In [5]:
leagues_ids = []

for name in tournaments:
    result = find_main_event_by_name(name, all_leagues)
    print(result)
    
    if not result.empty:
        leagues_ids.extend(result["leagueid"].tolist())
leagues_ids = list(set(leagues_ids))
print("Final League IDs:", leagues_ids)

      leagueid     name          tier
7729     17419  SLAM IV  professional
      leagueid                  name          tier
8921     18863  FISSURE PLAYGROUND 2  professional
      leagueid                         name          tier
8964     18920  PGL Wallachia 2025 Season 6  professional
      leagueid           name          tier
7730     17420         SLAM V  professional
9114     19099  BLAST SLAM VI  professional
      leagueid                   name          tier
9020     18988  DreamLeague Season 27  professional
      leagueid           name          tier
9114     19099  BLAST SLAM VI  professional
      leagueid                   name          tier
9253     19269  DreamLeague Season 28  professional
Final League IDs: [19269, 18920, 17419, 17420, 18988, 18863, 19099]


In [6]:
test_id = leagues_ids[0]
df = get_league_matches(test_id)
print(df.shape)

(195, 13)


In [7]:
all_matches = []

for lid in leagues_ids:
    df = get_league_matches(lid)
    print(f"League {lid}: {df.shape}")
    all_matches.append(df)
    time.sleep(1)
    
match_df = pd.concat(all_matches, ignore_index=True)
print("Total Matches:", match_df.shape)    

League 19269: (195, 13)
League 18920: (118, 13)
League 17419: (96, 13)
League 17420: (94, 13)
League 18988: (206, 13)
League 18863: (124, 13)
League 19099: (100, 13)
Total Matches: (933, 13)


In [8]:
match_df.to_csv("../data/raw/raw_matches.csv", index=False)

In [9]:
df = pd.read_csv("../data/processed/clean_matches.csv")

In [10]:
teams =  pd.concat([
    df["radiant_team_id"],
    df["dire_team_id"]
]).dropna().unique()

print(len(teams))

35


In [11]:
team_map = {}

for team_id in teams:
    data = get_team_info(int(team_id))
    team_map[team_id] = data.get("name")

    time.sleep(1)

In [12]:
df["radiant_team_name"] = df["radiant_team_id"].map(team_map)
df["dire_team_name"] = df["dire_team_id"].map(team_map)

In [13]:
team_df = pd.DataFrame(
    team_map.items(),
    columns=["team_id", "team_name"]
)

team_df.head()

,team_id,team_name
0,8291895,Tundra Esports
1,9467224,Aurora Gaming
2,2163,Team Liquid
3,8261500,Xtreme Gaming
4,9338413,MOUZ


In [14]:
team_df.to_csv("../data/processed/team_mapping.csv", index=False)

In [ ]:
match_df

,match_id,radiant_win,start_time,duration,leagueid,radiant_score,dire_score,radiant_team_id,radiant_team_name,dire_team_id,dire_team_name,series_id,series_type
0,8712056091,True,1772388652,2833,19269,32,31,8291895,None,9467224,None,1069888.0,2.0
1,8711970053,True,1772384823,1919,19269,37,8,9467224,None,8291895,None,1069888.0,2.0
2,8711885801,False,1772381334,1691,19269,6,28,9467224,None,8291895,None,1069888.0,2.0
3,8711768658,False,1772376899,2508,19269,11,37,9467224,None,8291895,None,1069888.0,2.0
4,8711578057,True,1772370355,2080,19269,30,15,9467224,None,2163,None,1069815.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
928,8675882674,False,1770117425,2410,19099,18,39,2163,None,2586976,None,1061169.0,0.0
929,8675848583,False,1770115260,1947,19099,6,23,9823272,None,9303484,None,1061166.0,0.0
930,8675821146,True,1770113347,2542,19099,32,19,7119388,None,36,None,1061164.0,0.0
931,8675768384,True,1770109234,4337,19099,37,34,8291895,None,9828897,None,1061159.0,0.0


In [20]:
match_detail = []
for md in match_df["match_id"]:
    detail_df = get_match_detail(md)
    match_detail.append(detail_df)
    time.sleep(1)
    
match_detail_df = pd.DataFrame(match_detail)

In [23]:
print(match_detail_df.shape)
print(match_detail_df.columns.tolist())

(933, 58)
['version', 'match_id', 'teamfights', 'pauses', 'objectives', 'chat', 'radiant_gold_adv', 'radiant_xp_adv', 'cosmetics', 'draft_timings', 'players', 'leagueid', 'start_time', 'duration', 'series_id', 'series_type', 'cluster', 'replay_salt', 'radiant_win', 'pre_game_duration', 'match_seq_num', 'tower_status_radiant', 'tower_status_dire', 'barracks_status_radiant', 'barracks_status_dire', 'first_blood_time', 'lobby_type', 'human_players', 'game_mode', 'flags', 'engine', 'radiant_score', 'dire_score', 'radiant_team_id', 'radiant_name', 'radiant_logo', 'radiant_team_complete', 'dire_team_id', 'dire_name', 'dire_logo', 'dire_team_complete', 'radiant_captain', 'dire_captain', 'picks_bans', 'od_data', 'league', 'radiant_team', 'dire_team', 'metadata', 'replay_url', 'patch', 'region', 'all_word_counts', 'my_word_counts', 'throw', 'loss', 'comeback', 'stomp']


In [24]:
match_detail_df.to_csv("../data/raw/match_detail.csv")